# MarketMirror: predicted brain response with TRIBE v2 (free Colab T4)

This notebook runs Meta's **TRIBE v2** research model on a free Colab GPU and gives MarketMirror a private web
address to send your ads to. MarketMirror then shows, second by second, how an *average* viewer's brain is
predicted to respond to each ad.

**License.** TRIBE v2 code and weights are **CC BY-NC 4.0: noncommercial use only**. Its text encoder,
`meta-llama/Llama-3.2-3B`, is gated: accept its license at https://huggingface.co/meta-llama/Llama-3.2-3B with
your own Hugging Face account before step 3. By running this notebook you accept both licenses.

**GPU.** Choose *Runtime > Change runtime type > T4 GPU* before running anything. The free T4 has limited
hours per day; the session (and the web address it makes) ends when Colab disconnects.

**What it means.** It predicts brain activity for an average viewer, not attention, feelings, memory or
buying. It is not validated for ads. Treat it as a research view next to the simulated buyers, never as a score.

## 1. Install TRIBE v2
This takes a few minutes. When it finishes, use *Runtime > Restart session*, then run the cells below (skip this one).

In [ ]:
!pip -q install uv
!uv pip install --system "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"

## 2. Sign in to Hugging Face
Paste your own read token in the box (from https://huggingface.co/settings/tokens). It is kept by Colab for this session; this notebook never prints or stores it.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Get the MarketMirror server
Downloads `tribe_server.py` from the repo. If the repo or branch is private or missing, the copy embedded below is used instead.

In [ ]:
import urllib.request, pathlib
REPO = "DarkWorldCoder/market-mirror"
BRANCH = "feat/codex-workspace-features"   # change to the branch you run MarketMirror from
url = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/backend/brain_sidecar/tribe_server.py"
EMBEDDED = "\"\"\"Small HTTP/JSON server that runs Meta's TRIBE v2 brain-response model and returns a compact summary.\n\nTRIBE v2 (https://huggingface.co/facebook/tribev2, code https://github.com/facebookresearch/tribev2) predicts how\nthe cortex of an *average* person responds to a video, audio or text stimulus, about once per second, on the\nfsaverage5 surface (~20k vertices). Its predictions are already shifted 5 s earlier to make up for the slow blood-flow\n(hemodynamic) response, so the seconds line up with the stimulus. It predicts brain activity only: no attention,\nfeelings, memory, words or buying.\n\nLicense: CC BY-NC 4.0 (noncommercial) for the code and the weights. The text encoder needs gated access to\nmeta-llama/Llama-3.2-3B. A GPU is needed in practice. This server is opt-in and never part of a default install.\n\nWhy a separate server: TRIBE needs torch, WhisperX, DINOv2, V-JEPA2 and a GPU, none of which belong in\n``backend/.venv``. It runs in ``backend/brain_sidecar/.venv`` (``brain_sidecar.sh setup``), in a free Colab T4\n(``colab/MarketMirror_TRIBE_v2.ipynb``) or in the owner's own private Hugging Face Space (``space/``).\nMarketMirror talks to it over HTTP (``app/services/brain_response.py``, ``MM_BRAIN_RESPONSE_URL``).\n\nAPI (JSON):\n\n* ``GET  /health`` -> ``{\"ok\": true, \"model\", \"license\", \"loaded\", \"device\", \"auth\"}`` (no token needed)\n* ``GET  /jobs/<id>`` -> ``{\"ok\": true, \"status\": \"running\"|\"done\"|\"failed\", \"summary\"?, \"error\"?}`` (token needed)\n* ``POST /predict`` with ``Authorization: Bearer <MM_TRIBE_TOKEN>`` (or ``X-MM-Tribe-Token: <MM_TRIBE_TOKEN>``,\n  for hosts such as a private Hugging Face Space whose Authorization header carries the HF token) when a token is\n  set, body one of\n  ``{\"kind\": \"video\"|\"audio\", \"data_b64\": \"...\", \"filename\": \"ad.mp4\"}`` (uploaded bytes),\n  ``{\"kind\": \"video\"|\"audio\"|\"text\", \"path\": \"/abs/file\"}`` (a file on this machine; off unless\n  ``MM_TRIBE_ALLOW_PATHS=1``) or ``{\"kind\": \"text\", \"text\": \"caption\"}``\n  -> ``{\"ok\": true, \"summary\": {...}}``. Add ``\"async\": true`` to get ``{\"ok\": true, \"job\": id}`` at once and poll\n  ``/jobs/<id>``: tunnels such as Cloudflare quick tunnels cut a request after about 100 s, and a video on a T4\n  can take longer. Raw vertices are never returned.\n\nSummary: per-second mean response of anatomical groups from nilearn's Destrieux surface atlas (fsaverage5):\nvisual areas (occipital), hearing areas (transverse and superior temporal), language areas (inferior frontal,\nangular / supramarginal, middle temporal) and the whole cortex; each z-scored within the clip, with its peak seconds,\nplus the duration, model id, license and the lag note.\n\nEnvironment: ``MM_TRIBE_HOST`` (default 127.0.0.1), ``MM_TRIBE_PORT`` (7860), ``MM_TRIBE_TOKEN`` (required for a\nnon-loopback host), ``MM_TRIBE_MODEL`` (facebook/tribev2), ``MM_TRIBE_CACHE`` (./cache), ``MM_TRIBE_MAX_MB`` (200),\n``MM_TRIBE_ALLOW_PATHS`` (0). Only the stdlib is imported at module level; numpy, nilearn and tribev2 load lazily,\nso tests can drive this file with a fake predictor.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport base64\nimport binascii\nimport hmac\nimport json\nimport os\nimport re\nimport sys\nimport tempfile\nimport threading\nimport traceback\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nfrom typing import Any, Callable, Dict, List, Optional, Sequence, Tuple\n\nMODEL_ID = os.environ.get(\"MM_TRIBE_MODEL\", \"facebook/tribev2\")\nLICENSE = \"CC BY-NC 4.0 (noncommercial)\"\nLAG_NOTE = (\"TRIBE v2 shifts its predictions 5 s earlier to make up for the slow blood-flow response, so the seconds \"\n            \"line up with the ad; treat each peak as accurate to a few seconds.\")\nLABEL = (\"Predicted for an average viewer by Meta's TRIBE v2 research model (noncommercial license). It predicts brain \"\n         \"activity, not attention, feelings, memory or buying. Not validated for ads.\")\nPARCELLATION = \"Destrieux surface atlas (nilearn, fsaverage5)\"\nKINDS = (\"video\", \"audio\", \"text\")\nSUFFIX = {\"video\": (\".mp4\", \".mov\", \".webm\", \".mkv\", \".avi\", \".m4v\"),\n          \"audio\": (\".wav\", \".mp3\", \".m4a\", \".flac\", \".ogg\", \".aac\"),\n          \"text\": (\".txt\",)}\n\n# Destrieux label prefixes per group (names as nilearn spells them, without the hemisphere).\nGROUPS: Dict[str, Dict[str, Any]] = {\n    \"visual\": {\"label\": \"visual areas\", \"prefixes\": (\n        \"G_cuneus\", \"G_occipital_middle\", \"G_occipital_sup\", \"G_and_S_occipital_inf\", \"G_oc-temp_med-Lingual\",\n        \"Pole_occipital\", \"S_calcarine\", \"S_oc_middle_and_Lunatus\", \"S_oc_sup_and_transversal\", \"S_occipital_ant\",\n        \"S_oc-temp_med_and_Lingual\")},\n    \"hearing\": {\"label\": \"hearing areas\", \"prefixes\": (\n        \"G_temp_sup-G_T_transv\", \"S_temporal_transverse\", \"G_temp_sup-Plan_tempo\", \"G_temp_sup-Lateral\")},\n    \"language\": {\"label\": \"language areas\", \"prefixes\": (\n        \"G_front_inf-Opercular\", \"G_front_inf-Triangul\", \"G_front_inf-Orbital\", \"G_pariet_inf-Angular\",\n        \"G_pariet_inf-Supramar\", \"G_temporal_middle\")},\n    \"cortex\": {\"label\": \"whole cortex\", \"prefixes\": None},\n}\nNON_CORTEX = (\"Unknown\", \"Medial_wall\")\n\n\ndef _env_flag(name: str) -> bool:\n    return os.environ.get(name, \"\").strip().lower() in (\"1\", \"true\", \"yes\", \"on\")\n\n\n# ---------------------------------------------------------------------------\n# Parcellation and summary (pure; numpy only)\n# ---------------------------------------------------------------------------\n\ndef _label_name(raw: Any) -> str:\n    return raw.decode(\"utf-8\", \"replace\") if isinstance(raw, (bytes, bytearray)) else str(raw)\n\n\ndef group_vertices(labels: Sequence[Any], map_left: Sequence[int], map_right: Sequence[int],\n                   n_vertices: int) -> Tuple[Dict[str, List[int]], List[str]]:\n    \"\"\"Vertex indices per group, assuming TRIBE orders the left hemisphere first, then the right (fsaverage5:\n    10242 + 10242). Returns ({group: indices}, notes).\"\"\"\n    names = [_label_name(x) for x in labels]\n    left, right = list(map_left), list(map_right)\n    notes: List[str] = []\n    if n_vertices == len(left) + len(right):\n        parcel = left + right\n    elif n_vertices == len(left):\n        parcel = left\n        notes.append(\"Only one hemisphere's worth of vertices; the left-hemisphere atlas was used.\")\n    else:\n        notes.append(f\"The prediction has {n_vertices} vertices, not the {len(left) + len(right)} of fsaverage5; \"\n                     \"only the whole-cortex mean is reported.\")\n        return {\"cortex\": list(range(n_vertices))}, notes\n    out: Dict[str, List[int]] = {}\n    for key, spec in GROUPS.items():\n        if spec[\"prefixes\"] is None:\n            idx = [i for i, lab in enumerate(parcel) if 0 <= lab < len(names) and names[lab] not in NON_CORTEX]\n        else:\n            wanted = {j for j, n in enumerate(names) if n.startswith(spec[\"prefixes\"])}\n            idx = [i for i, lab in enumerate(parcel) if lab in wanted]\n        if idx:\n            out[key] = idx\n        else:\n            notes.append(f\"No atlas vertices matched {spec['label']}.\")\n    return out, notes\n\n\ndef _windows(z: Sequence[float], seconds: Sequence[float], threshold: float = 1.0, top: int = 3) -> List[Dict[str, Any]]:\n    \"\"\"Runs of seconds at or above `threshold` z, strongest first; the single top second when nothing clears it.\"\"\"\n    runs: List[Tuple[int, int]] = []\n    start = None\n    for i, v in enumerate(z):\n        if v is not None and v >= threshold:\n            start = i if start is None else start\n        elif start is not None:\n            runs.append((start, i - 1))\n            start = None\n    if start is not None:\n        runs.append((start, len(z) - 1))\n    if not runs and len(z):\n        best = max(range(len(z)), key=lambda i: z[i])\n        runs = [(best, best)]\n    out = []\n    for a, b in runs:\n        peak = max(range(a, b + 1), key=lambda i: z[i])\n        out.append({\"start\": float(seconds[a]), \"end\": float(seconds[b]), \"peak\": float(seconds[peak]),\n                    \"z\": round(float(z[peak]), 2)})\n    out.sort(key=lambda w: -w[\"z\"])\n    return out[:top]\n\n\ndef summarize(preds: Any, seconds: Optional[Sequence[float]], groups: Dict[str, List[int]], *,\n              kind: str = \"video\", notes: Optional[List[str]] = None, hz: float = 1.0) -> Dict[str, Any]:\n    \"\"\"Compact summary of a (n_timesteps, n_vertices) prediction: group means z-scored within the clip.\"\"\"\n    import numpy as np\n    arr = np.asarray(preds, dtype=float)\n    if arr.ndim != 2 or arr.shape[0] < 1:\n        raise ValueError(\"predictions must be a 2-D (timesteps, vertices) array\")\n    n_t, n_v = arr.shape\n    secs = [float(s) for s in (seconds if seconds is not None and len(seconds) == n_t else [i / hz for i in range(n_t)])]\n    step = (secs[1] - secs[0]) if n_t > 1 else 1.0 / hz\n    out_groups: Dict[str, Any] = {}\n    for key, idx in groups.items():\n        idx = [i for i in idx if 0 <= i < n_v]\n        if not idx:\n            continue\n        mean = arr[:, idx].mean(axis=1)\n        sd = float(mean.std())\n        z = (mean - mean.mean()) / sd if sd > 1e-12 else np.zeros_like(mean)\n        zl = [round(float(v), 3) for v in z]\n        out_groups[key] = {\"label\": GROUPS.get(key, {}).get(\"label\", key), \"vertices\": len(idx), \"z\": zl,\n                           \"raw_mean\": round(float(mean.mean()), 5), \"raw_sd\": round(sd, 5),\n                           \"peaks\": _windows(zl, secs)}\n    return {\"model\": MODEL_ID, \"license\": LICENSE, \"lag_note\": LAG_NOTE, \"label\": LABEL, \"parcellation\": PARCELLATION,\n            \"kind\": kind, \"duration_s\": round(secs[-1] + step, 2) if secs else 0.0,\n            \"hz\": round(1.0 / step, 3) if step else hz,\n            \"n_timesteps\": n_t, \"n_vertices\": n_v, \"seconds\": [round(s, 2) for s in secs], \"groups\": out_groups,\n            \"scale\": \"z-scored within this clip: 0 is the clip's own average, so compare shapes, not levels across ads\",\n            \"notes\": list(notes or [])}\n\n\ndef segment_seconds(segments: Any, n: int) -> Optional[List[float]]:\n    \"\"\"Start times from TRIBE's `segments` when they carry them (attribute or key start/onset); else None.\"\"\"\n    try:\n        items = list(segments)\n    except TypeError:\n        return None\n    if len(items) != n:\n        return None\n    out = []\n    for s in items:\n        v = None\n        for k in (\"start\", \"onset\", \"time\", \"t\"):\n            v = s.get(k) if isinstance(s, dict) else getattr(s, k, None)\n            if v is not None:\n                break\n        if v is None:\n            return None\n        try:\n            out.append(float(v))\n        except (TypeError, ValueError):\n            return None\n    return out\n\n\n# ---------------------------------------------------------------------------\n# The real model (lazy) and the atlas (lazy)\n# ---------------------------------------------------------------------------\n\nclass TribePredictor:\n    \"\"\"Loads TRIBE v2 on first use. predict(kind, path) -> (preds, seconds or None).\"\"\"\n\n    def __init__(self, model_id: str = MODEL_ID, cache: Optional[str] = None):\n        self.model_id = model_id\n        self.cache = cache or os.environ.get(\"MM_TRIBE_CACHE\", \"./cache\")\n        self._model = None\n        self._lock = threading.Lock()\n\n    @property\n    def loaded(self) -> bool:\n        return self._model is not None\n\n    def device(self) -> str:\n        try:\n            import torch\n            return \"cuda\" if torch.cuda.is_available() else \"cpu\"\n        except Exception:  # noqa: BLE001 - torch missing: not set up\n            return \"unknown\"\n\n    def _load(self):\n        if self._model is None:\n            from pathlib import Path\n            try:\n                from tribev2.demo_utils import TribeModel\n            except ImportError:\n                from tribev2 import TribeModel\n            self._model = TribeModel.from_pretrained(self.model_id, cache_folder=Path(self.cache))\n        return self._model\n\n    def predict(self, kind: str, path: str):\n        with self._lock:        # one GPU job at a time\n            model = self._load()\n            df = model.get_events_dataframe(**{f\"{kind}_path\": path})\n            preds, segments = model.predict(events=df)\n        try:\n            preds = preds.detach().cpu().numpy()\n        except AttributeError:\n            pass\n        return preds, segment_seconds(segments, len(preds))\n\n\n_ATLAS: Dict[str, Any] = {}\n\n\ndef destrieux() -> Tuple[List[Any], List[int], List[int]]:\n    if not _ATLAS:\n        from nilearn import datasets\n        atlas = datasets.fetch_atlas_surf_destrieux()\n        _ATLAS.update(labels=list(atlas[\"labels\"]), left=[int(x) for x in atlas[\"map_left\"]],\n                      right=[int(x) for x in atlas[\"map_right\"]])\n    return _ATLAS[\"labels\"], _ATLAS[\"left\"], _ATLAS[\"right\"]\n\n\n# ---------------------------------------------------------------------------\n# HTTP\n# ---------------------------------------------------------------------------\n\ndef _max_bytes() -> int:\n    try:\n        return int(float(os.environ.get(\"MM_TRIBE_MAX_MB\", \"200\")) * 1024 * 1024)\n    except ValueError:\n        return 200 * 1024 * 1024\n\n\ndef _safe_suffix(kind: str, filename: str) -> str:\n    ext = os.path.splitext(str(filename or \"\"))[1].lower()\n    return ext if ext in SUFFIX[kind] else SUFFIX[kind][0]\n\n\ndef run_predict(body: Dict[str, Any], predictor,\n                atlas: Callable[[], Tuple[List[Any], List[int], List[int]]]) -> Dict[str, Any]:\n    kind = str(body.get(\"kind\") or \"\").lower()\n    if kind not in KINDS:\n        raise ValueError(\"kind must be video, audio or text\")\n    tmp = None\n    try:\n        if body.get(\"path\"):\n            if not _env_flag(\"MM_TRIBE_ALLOW_PATHS\"):\n                raise ValueError(\"local paths are off on this server (MM_TRIBE_ALLOW_PATHS=1); upload the bytes instead\")\n            path = os.path.abspath(str(body[\"path\"]))\n            if not os.path.isfile(path) or not path.lower().endswith(SUFFIX[kind]):\n                raise ValueError(f\"no {kind} file at that path\")\n        elif kind == \"text\" and isinstance(body.get(\"text\"), str):\n            text = body[\"text\"].strip()\n            if not text:\n                raise ValueError(\"text is empty\")\n            fd, tmp = tempfile.mkstemp(suffix=\".txt\")\n            with os.fdopen(fd, \"w\", encoding=\"utf-8\") as f:\n                f.write(text[:20000])\n            path = tmp\n        elif isinstance(body.get(\"data_b64\"), str):\n            try:\n                data = base64.b64decode(body[\"data_b64\"], validate=True)\n            except (binascii.Error, ValueError):\n                raise ValueError(\"data_b64 is not valid base64\")\n            if not data:\n                raise ValueError(\"the upload is empty\")\n            if len(data) > _max_bytes():\n                raise ValueError(f\"upload larger than {_max_bytes() // (1024 * 1024)} MB\")\n            fd, tmp = tempfile.mkstemp(suffix=_safe_suffix(kind, body.get(\"filename\") or \"\"))\n            with os.fdopen(fd, \"wb\") as f:\n                f.write(data)\n            path = tmp\n        else:\n            raise ValueError(\"send data_b64 (+ filename), text, or path\")\n        preds, seconds = predictor.predict(kind, path)\n        labels, left, right = atlas()\n        n_v = len(preds[0]) if len(preds) else 0\n        groups, notes = group_vertices(labels, left, right, n_v)\n        if seconds is None:\n            notes.append(\"TRIBE gave no segment times; seconds assume 1 Hz from the start of the clip.\")\n        return summarize(preds, seconds, groups, kind=kind, notes=notes)\n    finally:\n        if tmp and os.path.exists(tmp):\n            os.unlink(tmp)\n\n\nclass Jobs:\n    \"\"\"Background predictions for clients behind a tunnel with a short request timeout. Keeps the last 50.\"\"\"\n\n    def __init__(self, keep: int = 50):\n        self.keep = keep\n        self._jobs: Dict[str, Dict[str, Any]] = {}\n        self._lock = threading.Lock()\n\n    def submit(self, fn: Callable[[], Dict[str, Any]]) -> str:\n        import secrets\n        job_id = secrets.token_hex(8)\n        with self._lock:\n            self._jobs[job_id] = {\"status\": \"running\"}\n            while len(self._jobs) > self.keep:\n                self._jobs.pop(next(iter(self._jobs)))\n\n        def work():\n            try:\n                result = {\"status\": \"done\", \"summary\": fn()}\n            except ValueError as e:\n                result = {\"status\": \"failed\", \"error\": str(e)}\n            except Exception as e:  # noqa: BLE001 - reported through the job\n                traceback.print_exc()\n                result = {\"status\": \"failed\", \"error\": f\"{type(e).__name__}: {str(e)[:300]}\"}\n            with self._lock:\n                if job_id in self._jobs:\n                    self._jobs[job_id] = result\n\n        threading.Thread(target=work, daemon=True).start()\n        return job_id\n\n    def get(self, job_id: str) -> Optional[Dict[str, Any]]:\n        with self._lock:\n            job = self._jobs.get(job_id)\n            return dict(job) if job else None\n\n\ndef make_handler(predictor, atlas, token: Optional[str]):\n    expected = (token or \"\").strip()\n    jobs = Jobs()\n\n    class Handler(BaseHTTPRequestHandler):\n        server_version = \"MarketMirrorTribe/1\"\n\n        def log_message(self, fmt, *args):  # short lines; headers (which carry the token) are never logged\n            sys.stderr.write(f\"tribe: {getattr(self, 'command', '')} {str(getattr(self, 'path', '')).split('?')[0][:120]}\\n\")\n\n        def _send(self, status: int, payload: Dict[str, Any]):\n            body = json.dumps(payload, ensure_ascii=False).encode(\"utf-8\")\n            self.send_response(status)\n            self.send_header(\"Content-Type\", \"application/json\")\n            self.send_header(\"Content-Length\", str(len(body)))\n            self.send_header(\"Cache-Control\", \"no-store\")\n            self.end_headers()\n            self.wfile.write(body)\n\n        def _authorized(self) -> bool:\n            if not expected:\n                return True\n            # X-MM-Tribe-Token for hosts whose Authorization header carries their own token (a private HF Space)\n            candidates = [self.headers.get(\"X-MM-Tribe-Token\") or \"\"]\n            m = re.match(r\"(?i)bearer\\s+(.+)\", (self.headers.get(\"Authorization\") or \"\").strip())\n            if m:\n                candidates.append(m.group(1))\n            return any(c.strip() and hmac.compare_digest(c.strip().encode(), expected.encode()) for c in candidates)\n\n        def do_GET(self):  # noqa: N802\n            route = self.path.split(\"?\")[0]\n            if route.startswith(\"/jobs/\"):\n                if not self._authorized():\n                    return self._send(401, {\"ok\": False, \"error\": \"missing or wrong bearer token\"})\n                job = jobs.get(route[len(\"/jobs/\"):])\n                return self._send(200, {\"ok\": True, **job}) if job else self._send(404, {\"ok\": False, \"error\": \"no such job\"})\n            if route != \"/health\":\n                return self._send(404, {\"ok\": False, \"error\": \"not found\"})\n            self._send(200, {\"ok\": True, \"model\": predictor.model_id, \"license\": LICENSE, \"loaded\": predictor.loaded,\n                             \"device\": predictor.device(), \"auth\": bool(expected), \"label\": LABEL})\n\n        def do_POST(self):  # noqa: N802\n            if self.path.split(\"?\")[0] != \"/predict\":\n                return self._send(404, {\"ok\": False, \"error\": \"not found\"})\n            if not self._authorized():\n                return self._send(401, {\"ok\": False, \"error\": \"missing or wrong bearer token\"})\n            try:\n                length = int(self.headers.get(\"Content-Length\") or 0)\n            except ValueError:\n                length = 0\n            if length <= 0:\n                return self._send(400, {\"ok\": False, \"error\": \"empty body\"})\n            if length > _max_bytes() * 4 // 3 + 4096:\n                return self._send(413, {\"ok\": False, \"error\": f\"upload larger than {_max_bytes() // (1024 * 1024)} MB\"})\n            try:\n                body = json.loads(self.rfile.read(length).decode(\"utf-8\"))\n                if not isinstance(body, dict):\n                    raise ValueError(\"body must be a JSON object\")\n                if body.get(\"async\"):\n                    return self._send(202, {\"ok\": True, \"job\": jobs.submit(lambda: run_predict(body, predictor, atlas))})\n                summary = run_predict(body, predictor, atlas)\n            except ValueError as e:\n                return self._send(400, {\"ok\": False, \"error\": str(e)})\n            except Exception as e:  # noqa: BLE001 - report, keep serving\n                traceback.print_exc()\n                return self._send(500, {\"ok\": False, \"error\": f\"{type(e).__name__}: {str(e)[:300]}\"})\n            self._send(200, {\"ok\": True, \"summary\": summary})\n\n    return Handler\n\n\ndef _loopback(host: str) -> bool:\n    return host in (\"127.0.0.1\", \"localhost\", \"::1\")\n\n\ndef make_server(host: str = \"127.0.0.1\", port: int = 7860, predictor=None, atlas=None,\n                token: Optional[str] = None) -> ThreadingHTTPServer:\n    token = os.environ.get(\"MM_TRIBE_TOKEN\", \"\") if token is None else token\n    if not _loopback(host) and not token:\n        raise SystemExit(\"refusing to listen on a non-loopback host without MM_TRIBE_TOKEN\")\n    return ThreadingHTTPServer((host, port), make_handler(predictor or TribePredictor(), atlas or destrieux, token))\n\n\ndef main() -> None:\n    host = os.environ.get(\"MM_TRIBE_HOST\", \"127.0.0.1\")\n    port = int(os.environ.get(\"MM_TRIBE_PORT\", \"7860\"))\n    server = make_server(host, port)\n    print(json.dumps({\"ok\": True, \"url\": f\"http://{host}:{port}\", \"model\": MODEL_ID, \"license\": LICENSE,\n                      \"auth\": bool(os.environ.get(\"MM_TRIBE_TOKEN\"))}), flush=True)\n    server.serve_forever()\n\n\nif __name__ == \"__main__\":\n    main()\n"
try:
    source = urllib.request.urlopen(url, timeout=30).read().decode("utf-8")
    print("downloaded", url)
except Exception as e:
    source = EMBEDDED
    print("using the embedded copy (", type(e).__name__, ")")
pathlib.Path("tribe_server.py").write_text(source, encoding="utf-8")

## 4. Start the server
Makes a fresh access token for this session and starts the server on `127.0.0.1:7860` (only reachable inside this Colab machine until step 5). The model weights (about 1 GB plus the encoders) download on the first prediction.

In [ ]:
import os, secrets, subprocess, sys, time, json, urllib.request
MM_TRIBE_TOKEN = secrets.token_urlsafe(32)
env = dict(os.environ, MM_TRIBE_TOKEN=MM_TRIBE_TOKEN, MM_TRIBE_HOST="127.0.0.1", MM_TRIBE_PORT="7860",
           MM_TRIBE_CACHE="/content/tribe_cache")
server = subprocess.Popen([sys.executable, "tribe_server.py"], env=env,
                          stdout=open("tribe_server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        health = json.loads(urllib.request.urlopen("http://127.0.0.1:7860/health", timeout=2).read())
        print("server is up:", {k: health[k] for k in ("model", "license", "device", "auth")})
        break
    except Exception:
        if server.poll() is not None:
            print(open("tribe_server.log").read()[-2000:]); raise SystemExit("the server stopped")
        time.sleep(1)
else:
    raise SystemExit("the server did not answer; see tribe_server.log")

## 5. Open a temporary tunnel
Downloads `cloudflared` from Cloudflare's official GitHub releases and opens a quick tunnel to the server. Anyone with the address still needs the token from step 4.

In [ ]:
import re, subprocess, time, urllib.request, os
CF = "/content/cloudflared"
if not os.path.exists(CF):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    os.chmod(CF, 0o755)
tunnel = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:7860"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
PUBLIC_URL = None
deadline = time.time() + 60
while time.time() < deadline and PUBLIC_URL is None:
    line = tunnel.stdout.readline()
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line or "")
    if m:
        PUBLIC_URL = m.group(0)
if not PUBLIC_URL:
    raise SystemExit("no tunnel address yet; run this cell again")
print("tunnel:", PUBLIC_URL)

## 6. Connect MarketMirror
Copy the two lines below into MarketMirror's `.env` (next to `LLM_API_KEY`) and restart the backend. The token is shown only to you, in this notebook. The tunnel and the token end when this Colab session ends; run steps 4 to 6 again next time.

In [ ]:
print("MM_BRAIN_RESPONSE_URL=" + PUBLIC_URL)
print("MM_BRAIN_RESPONSE_TOKEN=" + MM_TRIBE_TOKEN)

## 7. Optional: smoke test on the Sintel trailer
Sends the CC-BY Sintel trailer (Blender Foundation) to the local server and plots the four groups. The first run downloads the weights, so it can take several minutes.

In [ ]:
import base64, json, time, urllib.request
import matplotlib.pyplot as plt
clip = "/content/sintel_trailer-480p.mp4"
if not os.path.exists(clip):
    urllib.request.urlretrieve("https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4", clip)
headers = {"Authorization": "Bearer " + MM_TRIBE_TOKEN, "Content-Type": "application/json"}
body = json.dumps({"kind": "video", "filename": "sintel.mp4", "async": True,
                   "data_b64": base64.b64encode(open(clip, "rb").read()).decode()}).encode()
job = json.loads(urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:7860/predict", data=body, headers=headers)).read())["job"]
while True:
    state = json.loads(urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:7860/jobs/" + job, headers=headers)).read())
    if state["status"] != "running":
        break
    time.sleep(5)
if state["status"] != "done":
    raise SystemExit(state.get("error"))
s = state["summary"]
plt.figure(figsize=(10, 3.5))
for key, g in s["groups"].items():
    plt.plot(s["seconds"], g["z"], label=g["label"])
plt.axhline(0, color="#999", lw=0.6)
plt.xlabel("seconds into the clip"); plt.ylabel("z within clip"); plt.legend(); plt.title("Predicted response, average viewer (TRIBE v2)")
plt.show()
print(s["label"]); print(s["lag_note"])
for key, g in s["groups"].items():
    print(g["label"], "peaks:", [(p["start"], p["end"]) for p in g["peaks"]])